In [1]:
%%bash

apt install pciutils lshw zstd
curl -fsSL https://ollama.com/install.sh | sh

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  libpci3 pci.ids usb.ids
The following NEW packages will be installed:
  libpci3 lshw pci.ids pciutils usb.ids zstd
0 upgraded, 6 newly installed, 0 to remove and 42 not upgraded.
Need to get 1,486 kB of archives.
After this operation, 4,951 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 pci.ids all 0.0~2022.01.22-1ubuntu0.1 [251 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libpci3 amd64 1:3.7.0-6 [28.9 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 lshw amd64 02.19.git.2021.06.19.996aaad9c7-2build1 [321 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/main amd64 pciutils amd64 1:3.7.0-6 [63.6 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy/main amd64 usb.ids all 2022.04.02-1 [219 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3bui



>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
%%bash

wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
chmod +x cloudflared-linux-amd64
mv cloudflared-linux-amd64 /usr/local/bin/cloudflared
cloudflared --version

cloudflared version 2026.3.0 (built 2026-03-09-14:08 UTC)


In [ ]:
import os
import subprocess
import time

OLLAMA_MODEL = "gemma3:latest"
OLLAMA_MODEL = "qwen3-vl:latest"

print("Starting Ollama server...")
process = subprocess.Popen(["ollama", "serve"],
                           stdout=subprocess.DEVNULL,
                           stderr=subprocess.DEVNULL)

time.sleep(10)

Starting Ollama server...


In [6]:
print(f"Pulling model {OLLAMA_MODEL} from Ollama ...")
!ollama pull $OLLAMA_MODEL

Pulling model qwen3-vl:latest from Ollama ...



In [7]:
print(f"Pulling model nomic-embed-text:latest from Ollama ...")
!ollama pull nomic-embed-text:latest

Pulling model nomic-embed-text:latest from Ollama ...



In [8]:
print(f"Pulling model llama3.2:latest from Ollama ...")
!ollama pull llama3.2:latest

Pulling model llama3.2:latest from Ollama ...



In [10]:
!ollama list

NAME                       ID              SIZE      MODIFIED               
llama3.2:latest            a80c4f17acd5    2.0 GB    Less than a second ago    
nomic-embed-text:latest    0a109f422b47    274 MB    24 seconds ago            
qwen3-vl:latest            901cae732162    6.1 GB    32 seconds ago            


In [11]:
print("Starting Ollama server...")
process = subprocess.Popen(["ollama", "run", OLLAMA_MODEL],
                           stdout=subprocess.DEVNULL,
                           stderr=subprocess.DEVNULL)


Starting Ollama server...


In [ ]:
#@title Generate public link throw cloudflared tunnel

!cloudflared tunnel --url http://localhost:11434 --http-host-header="localhost:11434"


2026-04-22T06:39:41Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-04-22T06:39:41Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-04-22T06:39:43Z INF +--------------------------------------------------------------------------------------------+
2026-04-22T06:39:43Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-04-22T06:39:43Z INF |  https://seed-membrane-jewish-boundaries.trycloudflare

In [ ]:
# OLLAMA_URL = "https://afterwards-feature-bases-eddie.trycloudflare.com"

In [ ]:
import requests

def query_ollama(prompt, model="qwen3:latest"):
    data = {
        "prompt": prompt,
        "model": model,
        "stream": False
    }

    response = requests.post(f"{OLLAMA_URL}/api/generate", json=data)

    if response.status_code == 200:
        return response.json().get("response", "No response Found")
    else:
        return f"Error: {response.status_code}, {response.text}"


In [ ]:
%%time

response = query_ollama("calcualte this 131223 + 23143 = ? without resoning",
                        model=OLLAMA_MODEL)
print(response)

In [ ]:
import time

while True:
    time.sleep(60)